# R3-00 · Onramp express — Profundización (opcional) 🔬

**Formación Pública — Rama R3 · IA Aplicada · Notebook de profundización**

Opcional y de **transferencia**: lo mismo de la lección (dicts, funciones, JSON), pero con otro
dato real — la **calidad del aire** diaria (PM2,5) de 14 estaciones del SINCA, 2020-2025. Vas a
convertir una tabla en una **respuesta tipo API** y a leer un JSON anidado.

> Cada ejercicio termina en una **celda de chequeo** con ✅ / pista.

In [ ]:
import os, json, urllib.request
import pandas as pd

# "En vivo o caché": usa los archivos locales; si faltan (ej. Colab), se bajan del repo.
for archivo in ["aire_diario.csv", "estaciones.csv"]:
    if not os.path.exists(archivo):
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/formacionpublica-bootcamp/bootcamp/main/data/{archivo}", archivo)

aire = pd.read_csv("aire_diario.csv")          # PM2,5 diario por estación (SINCA, 2020-2025)
estaciones = pd.read_csv("estaciones.csv")     # dónde está cada estación
print("aire:", aire.shape, "| estaciones:", estaciones.shape)
aire.head(3)

## 1. De tabla a lista de dicts (y un dict por comprensión)

`df.to_dict("records")` convierte una tabla en una **lista de diccionarios** — exactamente la forma de un JSON. Con un **dict por comprensión** armas un «buscador» estación → región.

### ✍️ Ejercicio 1 — Construye el buscador `region_de`

In [ ]:
registros = estaciones.to_dict("records")
# TODO: dict por comprensión {estacion: region} recorriendo registros
region_de = ...
print(region_de)

In [ ]:
# ── Celda de chequeo · Ejercicio 1 ──────────────────────────────────────────
try:
    assert isinstance(registros, list) and isinstance(registros[0], dict)
    assert len(region_de) == len(estaciones)
    assert region_de["Nielol"] == "Región de La Araucanía"
    print("✅ Ejercicio 1: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", "{r['estacion']: r['region'] for r in registros}.")
    print("   Detalle:", e)

## 2. Números que llegan sucios

Las respuestas de las APIs traen números como texto (`"16.0"`), vacíos o marcas como `"s/d"` (sin dato). Una función **defensiva** nunca se cae: devuelve `None` cuando no hay número. Ojo: `float("nan")` NO lanza error — el truco es que `NaN` no es igual a sí mismo.

### ✍️ Ejercicio 2 — Escribe `a_numero`

In [ ]:
def a_numero(x):
    # TODO: intenta float(x); si lanza ValueError o TypeError, devuelve None
    ...
    # TODO: si el resultado es NaN (v != v), devuelve None; si no, el número

print(a_numero("16.0"), a_numero("s/d"), a_numero(None), a_numero(aire["pm25"].iloc[0]))

In [ ]:
# ── Celda de chequeo · Ejercicio 2 ──────────────────────────────────────────
try:
    assert a_numero("16.0") == 16.0
    assert a_numero("s/d") is None and a_numero(None) is None
    assert a_numero(float("nan")) is None
    assert sum(a_numero(x) is None for x in aire["pm25"]) == int(aire["pm25"].isna().sum())
    print("✅ Ejercicio 2: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'try: v = float(x) / except (ValueError, TypeError): return None / if v != v: return None / return v.')
    print("   Detalle:", e)

## 3. Tu propia respuesta «tipo API»

Empaqueta el resumen de una estación en un mes como un dict listo para `json.dumps`: solo tipos nativos de Python (`int`, `float`, `str`). Como umbral usamos **50 µg/m³**, el valor de la norma diaria chilena de PM2,5.

### ✍️ Ejercicio 3 — Escribe `resumen_estacion`

In [ ]:
def resumen_estacion(aire, estacion, mes):
    serie = aire[(aire["estacion"] == estacion) & (aire["fecha"].str.startswith(mes))]["pm25"]
    return {
        "estacion": estacion,
        "region": region_de[estacion],
        "mes": mes,
        # TODO: días con dato (serie.count(), envuelto en int)
        "dias_con_dato": ...,
        # TODO: promedio de PM2,5 redondeado a 1 decimal (round(float(...), 1))
        "pm25_promedio": ...,
        # TODO: días sobre 50 µg/m³ (envuelto en int)
        "dias_sobre_50": ...,
    }

r = resumen_estacion(aire, "Las Encinas Temuco", "2024-06")
print(r)

In [ ]:
# ── Celda de chequeo · Ejercicio 3 ──────────────────────────────────────────
try:
    assert json.loads(json.dumps(r)) == r
    assert r["dias_con_dato"] == 30 and isinstance(r["dias_con_dato"], int)
    _s = aire[(aire["estacion"] == "Las Encinas Temuco") & (aire["fecha"].str.startswith("2024-06"))]["pm25"]
    assert r["pm25_promedio"] == round(float(_s.mean()), 1)
    assert r["dias_sobre_50"] == int((_s > 50).sum())
    print("✅ Ejercicio 3: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'int(serie.count()), round(float(serie.mean()), 1), int((serie > 50).sum()).')
    print("   Detalle:", e)

## 4. Leer un JSON anidado

Las APIs suelen responder con un sobre: metadatos arriba y una **lista de resultados** adentro. Lee la respuesta y encuentra la estación con el peor promedio de junio 2024.

### ✍️ Ejercicio 4 — Encuentra la peor estación en la respuesta

In [ ]:
respuesta = json.dumps({"fuente": "SINCA", "resultados": [
    resumen_estacion(aire, e, "2024-06") for e in ["Las Encinas Temuco", "Nielol", "Cerro Navia"]]})
# TODO: convierte la respuesta (texto) a dict
datos = ...
# TODO: el nombre de la estación con mayor "pm25_promedio" dentro de datos["resultados"]
peor = ...
print("estación con peor promedio:", peor)

In [ ]:
# ── Celda de chequeo · Ejercicio 4 ──────────────────────────────────────────
try:
    assert isinstance(datos, dict) and len(datos["resultados"]) == 3
    assert peor == max(datos["resultados"], key=lambda x: x["pm25_promedio"])["estacion"]
    assert peor == "Cerro Navia"
    print("✅ Ejercicio 4: ¡correcto!")
except Exception as e:
    print("❌ Aún no. Revisa tu respuesta y vuelve a intentarlo.")
    print("   Pista:", 'datos = json.loads(respuesta); max(datos["resultados"], key=lambda x: x["pm25_promedio"])["estacion"].')
    print("   Detalle:", e)

---

*Fin de la profundización de R3-00.*